# Experiment 01 — Baseline IDS on NSL-KDD

Frozen pre-MIA baseline comparing Random Forest, XGBoost, and MLP for binary Normal-vs-Attack classification. Thresholds are selected on a validation split using F2 and evaluated once on KDDTest+.


In [ ]:
import os, json, platform, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_DIR = Path(os.environ.get("ML_DP_NID_DIR", "/content/drive/MyDrive/ML-DP-NID"))
TRAIN_FILE = PROJECT_DIR / "KDDTrain+.txt"
TEST_FILE = PROJECT_DIR / "KDDTest+.txt"
RESULTS_DIR = PROJECT_DIR / "results" / "baseline_v1_pre_mia"
ARTIFACT_DIR = PROJECT_DIR / "artifacts" / "baseline_v1_pre_mia"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_FILE.exists() and TEST_FILE.exists()


In [ ]:
FEATURES = [
"duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
"wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
"root_shell","su_attempted","num_root","num_file_creations","num_shells",
"num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
"srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
"same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
"dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
"dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
"dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"
]
COLUMNS = FEATURES + ["label","difficulty"]
CAT = ["protocol_type","service","flag"]
NUM = [c for c in FEATURES if c not in CAT]

def load_data(path):
    df = pd.read_csv(path, names=COLUMNS)
    for c in CAT + ["label"]:
        df[c] = df[c].astype(str).str.strip().str.lower().str.rstrip(".")
    for c in NUM:
        df[c] = pd.to_numeric(df[c], errors="raise")
    df["binary_label"] = (df["label"] != "normal").astype(int)
    return df

train_df = load_data(TRAIN_FILE)
test_df = load_data(TEST_FILE)
train_part, val_part = train_test_split(
    train_df, test_size=0.20, random_state=SEED,
    stratify=train_df["binary_label"]
)


In [ ]:
def encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("num", MinMaxScaler(), NUM),
    ("cat", encoder(), CAT),
])

def xy(df):
    return df[FEATURES], df["binary_label"].to_numpy()

X_train_raw, y_train = xy(train_part)
X_val_raw, y_val = xy(val_part)
X_test_raw, y_test = xy(test_df)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)


In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=SEED, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective="binary:logistic", eval_metric="logloss",
        random_state=SEED, n_jobs=-1
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(64,32), activation="relu", solver="adam",
        alpha=1e-4, batch_size=256, learning_rate_init=1e-3,
        max_iter=30, random_state=SEED
    ),
}
for model in models.values():
    model.fit(X_train, y_train)


In [ ]:
def score(model, X):
    return model.predict_proba(X)[:,1]

def metrics(model_name, split, y_true, y_score, threshold, experiment):
    pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "Experiment": experiment, "Model": model_name, "Split": split,
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true,pred),
        "Precision": precision_score(y_true,pred,zero_division=0),
        "Recall": recall_score(y_true,pred,zero_division=0),
        "F1": f1_score(y_true,pred,zero_division=0),
        "F2": fbeta_score(y_true,pred,beta=2,zero_division=0),
        "False Negative Rate": fn/(fn+tp),
        "False Positive Rate": fp/(fp+tn),
        "ROC_AUC": roc_auc_score(y_true,y_score),
        "PR_AUC": average_precision_score(y_true,y_score),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

def choose_f2_threshold(y_true, y_score):
    rows=[]
    for t in np.arange(0.01,1.00,0.01):
        pred=(y_score>=t).astype(int)
        rows.append({"threshold":float(t),"f2":fbeta_score(y_true,pred,beta=2,zero_division=0)})
    table=pd.DataFrame(rows)
    best=float(table.loc[table["f2"].idxmax(),"threshold"])
    return best, table


In [ ]:
default_rows=[]
tuned_rows=[]
for name, model in models.items():
    val_score=score(model,X_val)
    test_score=score(model,X_test)
    default_rows.append(metrics(name,"Test",y_test,test_score,0.5,"Default threshold"))
    best_t, search_table=choose_f2_threshold(y_val,val_score)
    search_table.to_csv(RESULTS_DIR / f"{name.lower().replace(' ','_')}_threshold_search_validation.csv",index=False)
    tuned_rows.append(metrics(name,"Test",y_test,test_score,best_t,"Validation-tuned threshold"))

default_df=pd.DataFrame(default_rows)
tuned_df=pd.DataFrame(tuned_rows)
default_df.to_csv(RESULTS_DIR / "test_results_threshold_0_5.csv",index=False)
tuned_df.to_csv(RESULTS_DIR / "test_results_validation_tuned_threshold.csv",index=False)
pd.concat([default_df,tuned_df],ignore_index=True).to_csv(
    RESULTS_DIR / "default_vs_validation_tuned_threshold_comparison.csv",index=False
)
display(default_df)
display(tuned_df)


In [ ]:
joblib.dump(preprocessor, ARTIFACT_DIR / "nsl_kdd_preprocessor.joblib")
for name, model in models.items():
    joblib.dump(model, ARTIFACT_DIR / f"{name.lower().replace(' ','_')}.joblib")

manifest = {
    "experiment": "01_baseline_nsl_kdd_ids",
    "status": "frozen_pre_mia",
    "seed": SEED,
    "dataset": {"train": str(TRAIN_FILE), "test": str(TEST_FILE)},
    "split": {
        "train_rows": len(train_part),
        "validation_rows": len(val_part),
        "test_rows": len(test_df),
        "validation_fraction": 0.20,
    },
    "models": list(models),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
with open(ARTIFACT_DIR / "repro_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)
